In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.optim as optim

# Mini-Xception architecture definition
Three main parts : entry stem, residual blocks (with separable convolution) and a classification head.
## Entry stem definition

In [2]:
class EntryStem(nn.Module):
    def __init__(self, in_channels: int = 1, out_channels: int = 8):
        super().__init__()

        self.stem = nn.Sequential(
            # first convolution : goes from 1 channels (gray scale images) to 8 channels using 8 3x3 filters
            # the 8 output channels are of same dimension as the input images (48x48px)
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            # second convolution : we keep 8 channels using 8 filters with dimension 8x3x3 (3d masks that blend the channels)
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.stem(x)

## Residual block
The residual block has two branches. One uses separable convolution : first, each channel is convolved by a 3x3 filter (one per channel, we keep the same number of channels). This is called the "depthwise" step. Then, the resulting channels are blended using a 1D convolution as many times as needed to get the desired number of output channels. This is called the "pointwise" step.

In [3]:
class DepthwiseSeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, padding: int = 1):
        super().__init__()

        # "depthwise" convolution : each entry channel is convolved by a 3x3 mask, no mix between entry channels. We keep the same number of channels
        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            groups=in_channels,  # this option tells torch to not blend channels (2d convolution, not 3d)
            bias=False
        )

        # "pointwise" convolution : we blend channels using in_channels*1*1 masks, as many times as needed to obtain the desired number of output channels
        self.pointwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

We now define the residual block architecture. At each pass, it doubles the number of channels and halves the size of the picture.

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        # main branch : two separable conv
        self.main_branch = nn.Sequential(
            # first separable convolution - the number of channels goes from in_channels to out_channels
            DepthwiseSeparableConv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            # second separable convolution - the number of channels stays at out_channels
            DepthwiseSeparableConv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),  # no relu here

            # space dimension reduction via max pooling
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # residual branch ("shortcut"). uses a in_channels*1*1 convolution out_channels times with stride 2 to match the dimension of the main branch output (divides space dimension by two)
        self.shortcut = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,
                stride=2,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

        # after fusing the main and residual branches, we will have to apply relu
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # computing the outputs of the main branch and the residual branch, then adding them together, then relu
        residual = self.shortcut(x)
        out = self.main_branch(x)
        out = out + residual
        return self.relu(out)

## Classification head
This last block goes from 128 channels to 7 (we are sorting each picture in 7 emotions). (128,3,3) -> (7,3,3) using separable conv then average pooling to go from a 3x3 matrix for each emotion to a scalar.

In [5]:
class ClassificationHead(nn.Module):
    def __init__(self, in_channels: int = 128, num_classes: int = 7):
        super().__init__()

        # last separable convolution, we go from a lot of channels (128) to 7 channels (because 7 emotions to choose from)
        self.conv = nn.Sequential(
            DepthwiseSeparableConv2d(
                in_channels=in_channels,
                out_channels=num_classes,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(num_classes),
            nn.ReLU(inplace=True)
        )

        # average pooling : we need a scalar for each emotion, not a 3x3 matrix
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.gap(x)
        logits = torch.flatten(x, start_dim=1)
        return logits

## Full model assembly

In [6]:
class MiniXception(nn.Module):
    def __init__(self, in_channels: int = 1, num_classes: int = 7):
        super().__init__()

        # 1. entry stem : 1 -> 8 -> 8 (image size unchanged)
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 8, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True)
        )

        # 2. four consecutive residual blocks
        # (8 -> 16, 48x48 -> 24x24)
        self.block1 = ResidualBlock(in_channels=8, out_channels=16)
        # (16 -> 32, 24x24 -> 12x12)
        self.block2 = ResidualBlock(in_channels=16, out_channels=32)
        # (32 -> 64, 12x12 -> 6x6)
        self.block3 = ResidualBlock(in_channels=32, out_channels=64)
        # (64 -> 128, 6x6 -> 3x3)
        self.block4 = ResidualBlock(in_channels=64, out_channels=128)

        # 3. classification head
        self.head_conv = nn.Sequential(
            DepthwiseSeparableConv2d(128, num_classes, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_classes),
            nn.ReLU(inplace=True)
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.head_conv(x)
        x = self.gap(x)
        return torch.flatten(x, start_dim=1)

# Loading data & training

In [7]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomCrop(48, padding=4, padding_mode="edge"),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Ramène les pixels dans [-1, 1]
])

test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = ImageFolder(root="train", transform=train_transforms)
test_dataset = ImageFolder(root="test", transform=test_transforms)

print(f"Found classes : {train_dataset.classes}")
print(f"Training images : {len(train_dataset)} | Test images : {len(test_dataset)}")

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on : {device}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

Found classes : ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Training images : 28709 | Test images : 7178
Training on : cuda


In [8]:
model = MiniXception(in_channels=1, num_classes=len(train_dataset.classes)).to(device)

criterion = torch.nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=4
)

In [9]:
def run_epoch(model, dataloader, criterion, optimizer=None, is_train=True):
    model.train() if is_train else model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


EPOCHS = 60
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, is_train=True)
    val_loss, val_acc = run_epoch(model, test_loader, criterion, is_train=False)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Epoch [{epoch:02d}/{EPOCHS}] | "
          f"Train: Loss {train_loss:.4f}, Acc {train_acc*100:.2f}% | "
          f"Val: Loss {val_loss:.4f}, Acc {val_acc*100:.2f}% | LR: {current_lr:.1e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_mini_xception.pth")
        print(f"  -> Best model saved (Acc: {best_val_acc*100:.2f}%)")

Epoch [01/60] | Train: Loss 1.7729, Acc 31.65% | Val: Loss 1.5407, Acc 43.55% | LR: 1.0e-03
  -> Best model saved (Acc: 43.55%)
Epoch [02/60] | Train: Loss 1.5195, Acc 44.09% | Val: Loss 1.4059, Acc 47.95% | LR: 1.0e-03
  -> Best model saved (Acc: 47.95%)
Epoch [03/60] | Train: Loss 1.4022, Acc 48.13% | Val: Loss 1.3327, Acc 51.04% | LR: 1.0e-03
  -> Best model saved (Acc: 51.04%)


KeyboardInterrupt: 